# Tomofast-x-µ Minimal Example

This notebook demonstrates the basic workflow using the `tomofast_x_mu` Python package.

**Prerequisites:**
- Package installed: `pip install -e .`
- QDM `.mat` file available

**Note:** This example covers preprocessing only. To run actual inversions, you need Tomofast-x installed.

## Setup

In [ ]:
from pathlib import Path
import tomofast_x_mu as tfmu
import matplotlib.pyplot as plt

# Set up output directory
output_dir = Path("example_output")
output_dir.mkdir(exist_ok=True)

print(f"tomofast_x_mu package loaded successfully")
print(f"Output directory: {output_dir}")

## Step 1: Load QDM Data

Load a QDM `.mat` file and convert to CSV format.

**Important:** Use `h_units="auto"` to handle mixed-units convention (step in meters, h in micrometers).

In [ ]:
# Replace with your .mat file path
mat_path = Path("../real_qdm_data/NC07-002-2Bz_uc0.mat")

# Check if file exists
if not mat_path.exists():
    print(f"⚠️  File not found: {mat_path}")
    print("Please update mat_path to point to your QDM .mat file")
else:
    # Load .mat file -> xarray Dataset + CSV
    csv_path = output_dir / "qdm_data.csv"
    ds = tfmu.load_qdm_mat(mat_path, save_csv=True, csv_path=csv_path, h_units="auto")
    
    print(f"✓ Loaded QDM data")
    print(f"  Shape: {ds.bz.shape}")
    print(f"  Pixel size: {ds.attrs['pixel_size_um']:.2f} µm")
    print(f"  Sensor distance: {ds.attrs['sensor_sample_distance_um']:.2f} µm")
    print(f"  CSV saved to: {csv_path}")

## Step 2: Visualize Bz Field

Plot the vertical magnetic field with LED overlay (if available).

In [ ]:
if mat_path.exists():
    fig, ax = tfmu.plot_bz_led(mat_path, pix=ds.attrs['pixel_size_um'])
    ax.set_title(f"QDM Scan: {mat_path.name}")
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Bz field plotted")

## Step 3: Convert to Tomofast Format

Convert CSV to Tomofast observation file format.

**Note:** `flip_bz=True` converts from QDM convention (Z+ up) to Tomofast convention (Z+ down).

In [ ]:
if csv_path.exists():
    obs_path = output_dir / "Mag_Inversion.obs"
    
    result = tfmu.format_tomofast_obs(
        str(csv_path),
        str(obs_path),
        flip_bz=True  # QDM -> Tomofast convention
    )
    
    print(f"✓ Created Tomofast observation file")
    print(f"  Output: {obs_path}")
    print(f"  Grid size: {result['nx']} × {result['ny']}")
    print(f"  Total observations: {len(result['bz'])}")

## Step 4: Detect Magnetic Anomalies

Identify regions with significant magnetic anomalies ("blobs") to focus computational resources.

In [ ]:
if obs_path.exists():
    blob_obs_dir = output_dir / "blob_obs"
    windows_path = output_dir / "blob_windows.txt"
    summary_path = output_dir / "blob_summary.txt"
    
    blobs = tfmu.detect_anomalies(
        str(obs_path),
        threshold_factor=2.75,  # Threshold = 2.75 × std(Bz)
        pad_x=40,               # Padding cells in X
        pad_y=10,               # Padding cells in Y
        min_blob_cells=150,     # Minimum blob size
        max_blob_cells=100000,  # Maximum blob size
        nz_fixed=120,           # Fixed Z dimension
        out_obs_dir=str(blob_obs_dir),
        windows_path=str(windows_path),
        summary_path=str(summary_path)
    )
    
    print(f"✓ Detected {len(blobs)} magnetic anomalies")
    print(f"  Blob observation files: {blob_obs_dir}")
    print(f"  Windows file: {windows_path}")
    print(f"  Summary file: {summary_path}")
    
    # Show blob details
    for i, blob in enumerate(blobs[:3], 1):  # Show first 3
        print(f"\n  Blob {blob['blob_id']}:")
        print(f"    Size: {blob['nx_win']} × {blob['ny_win']} × {blob['nz_win']}")
        print(f"    Data points: {blob['ndata']}")

## Step 5: Generate Meshes

Create 3D meshes for each detected anomaly with adaptive padding.

In [ ]:
if windows_path.exists():
    mesh_dir = output_dir / "meshgrids"
    mesh_summary = output_dir / "mesh_summary.txt"
    
    meshes = tfmu.build_mesh_per_blob(
        str(windows_path),
        str(mesh_dir),
        str(mesh_summary),
        npad_x=20,      # Padding cells in X
        npad_y=20,      # Padding cells in Y
        nz=120,         # Number of Z cells
        dz0=4.4         # First Z cell thickness (µm)
    )
    
    print(f"✓ Generated {len(meshes)} mesh files")
    print(f"  Mesh directory: {mesh_dir}")
    print(f"  Mesh summary: {mesh_summary}")

## Step 6: Create Tomofast Parameter Files

Generate parameter files for Tomofast-x inversions.

In [ ]:
if mesh_summary.exists():
    par_dir = output_dir / "parfiles"
    
    parfiles = tfmu.create_inversion_parfiles(
        str(mesh_summary),
        str(par_dir)
    )
    
    print(f"✓ Created {len(parfiles)} parameter files")
    print(f"  Parfile directory: {par_dir}")
    
    # Show first parfile
    if parfiles:
        print(f"\n  Example: {Path(parfiles[0]).name}")

## Summary

You've successfully completed the preprocessing workflow!

**What was created:**
1. CSV file with QDM data
2. Tomofast observation file
3. Per-blob observation files
4. Mesh files for each anomaly
5. Parameter files for Tomofast-x

**Next steps:**
- To run inversions, you need Tomofast-x installed
- Use `tfmu.run_tomofast_blobs()` to execute inversions
- See `DOCUMENTATION.md` for the complete workflow including post-processing

**Files created in `example_output/`:**

In [ ]:
# List all created files
import os

print("\nCreated files:")
for root, dirs, files in os.walk(output_dir):
    level = root.replace(str(output_dir), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Show first 5 files per directory
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

## Optional: Run Inversions

If you have Tomofast-x installed, uncomment and run the following cell.

**Requirements:**
- Tomofast-x executable (`tomofastx`)
- MPI runtime (OpenMPI, MPICH, etc.)

**Note:** This will take significant time depending on the number of blobs and your hardware.

In [ ]:
# Uncomment to run inversions (requires Tomofast-x)

# output_dirs = tfmu.run_tomofast_blobs(
#     base_dir=str(output_dir.parent),
#     tomofast_home="/path/to/Tomofast-x",  # Update this path
#     nproc=7,
#     wsl_exe=None  # or r"C:\Windows\System32\wsl.exe" for WSL
# )
# 
# print(f"✓ Inversions complete")
# print(f"  Output directories: {len(output_dirs)}")